In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from datasets import load_dataset
dataset_name = "ridwanFatur98/synth-invoice"
train_dataset = load_dataset(dataset_name, split="train")
sample = train_dataset[0]

In [3]:
from scripts.image_process import process_image, image_to_tensor
from scripts.ground_truth_process import process_ground_truth, get_train_inputs
from scripts.donut_model import DonutModel
from scripts.tokenizer import get_tokenizer
tokenizer = get_tokenizer()

W0930 11:49:41.982000 103781 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0930 11:49:42.010000 103781 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [4]:
# Encoder Config
encoder_layer = [1, 1, 1, 1]
window_size = 10
input_size = [1280, 960]

# Decoder Config
max_position_embeddings = 768
decoder_layer = 1

In [5]:
donut_model = DonutModel(
    tokenizer,
    input_size, window_size, encoder_layer,
    max_position_embeddings, decoder_layer    
).to(device)

state_dict = torch.load(
    "donut_model.pth",
    map_location=device,
)

donut_model.load_state_dict(state_dict)
donut_model.eval()
pass

### Inference

In [6]:
from torch.nn.utils.rnn import pad_sequence
from transformers.file_utils import ModelOutput

In [7]:
import re
def token2json(tokens, is_inner_value=False):
    output = dict()

    while tokens:
        start_token = re.search(r"<s_(.*?)>", tokens, re.IGNORECASE)
        if start_token is None:
            break
        key = start_token.group(1)
        end_token = re.search(fr"</s_{key}>", tokens, re.IGNORECASE)
        start_token = start_token.group()
        if end_token is None:
            tokens = tokens.replace(start_token, "")
        else:
            end_token = end_token.group()
            start_token_escaped = re.escape(start_token)
            end_token_escaped = re.escape(end_token)
            content = re.search(f"{start_token_escaped}(.*?){end_token_escaped}", tokens, re.IGNORECASE)
            if content is not None:
                content = content.group(1).strip()
                if r"<s_" in content and r"</s_" in content:  # non-leaf node
                    value = token2json(content, is_inner_value=True)
                    if value:
                        if len(value) == 1:
                            value = value[0]
                        output[key] = value
                else:  # leaf nodes
                    output[key] = []
                    for leaf in content.split(r"<sep/>"):
                        leaf = leaf.strip()
                        if (
                            leaf in tokenizer.get_added_vocab()
                            and leaf[0] == "<"
                            and leaf[-2:] == "/>"
                        ):
                            leaf = leaf[1:-2]  # for categorical special tokens
                        output[key].append(leaf)
                    if len(output[key]) == 1:
                        output[key] = output[key][0]

            tokens = tokens[tokens.find(end_token) + len(end_token) :].strip()
            if tokens[:6] == r"<sep/>":  # non-leaf nodes
                return [output] + token2json(tokens[6:], is_inner_value=True)

    if len(output):
        return [output] if is_inner_value else output
    else:
        return [] if is_inner_value else {"text_sequence": tokens}


In [8]:
def inference(model, tokenizer, image_tensors, prompt_tensors):
    last_hidden_state = model.encoder(image_tensors)
    encoder_outputs = ModelOutput(last_hidden_state=last_hidden_state.flatten(1, 2), attentions=None)
    decoder_output = donut_model.decoder.model.generate(
        decoder_input_ids=prompt_tensors,
        encoder_outputs=encoder_outputs,
        max_length=max_length,
        early_stopping=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
        num_beams=1,
        bad_words_ids=[[tokenizer.unk_token_id]],
        return_dict_in_generate=True,
        output_attentions=False,
    )  
    output = {"predictions": list()}
    for seq in tokenizer.batch_decode(decoder_output.sequences):
        seq = seq.replace(tokenizer.eos_token, "").replace(tokenizer.pad_token, "")
        seq = re.sub(r"<.*?>", "", seq, count=1).strip()  # remove first task start token
        output["predictions"].append(token2json(seq))
    return output

In [10]:
# Input Config
input_size = [1280, 960]
align_long_axis = False
random_padding = True

# Input Encoder
img = process_image(sample["image"], input_size, align_long_axis, random_padding)
image_tensors = image_to_tensor(img)
image_tensors = image_tensors.unsqueeze(0)
image_tensors = image_tensors.to(device)

In [23]:
# Input Decoder
prompts = '<s_cord-v2>'
prompt_tensors = tokenizer(
    prompts,
    add_special_tokens=False,
    truncation=True,
    return_tensors="pt",
)["input_ids"].to(device)

In [24]:
prompt_tensors

tensor([[57567]], device='cuda:0')

In [25]:
result = inference(donut_model, tokenizer, image_tensors, prompt_tensors)

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [26]:
result

{'predictions': [{'bill_to': {'name': 'PT Maju Teknologi',
    'address': 'Gg. Tebet Barat Dalam No. 40',
    'city': 'Bontang, Indonesia',
    'phone': '+62 823-9935-2424',
    'email': 'finance@majuteknologi.com'},
   'invoice_date': '1 February 2025',
   'invoice_no': 'INV-2025-30496',
   'from_to': {'name': 'PT Solusi Nusantara',
    'address': 'Gang Antapani Lama No. 34',
    'city': 'Meulaboh, Indonesia',
    'phone': '+62 885-7912-1520',
    'email': 'billing@solusinusantara.com'},
   'items': [{'description': 'DevOps Services',
     'qty': '1',
     'price': '3800000',
     'total': '3800000'},
    {'description': 'UI/UX Design',
     'qty': '9',
     'price': '2900000',
     'total': '26100000'},
    {'description': 'UI/UX Design',
     'qty': '8',
     'price': '4000000',
     'total': '32000000'},
    {'description': 'Cloud Infrastructure',
     'qty': '1',
     'price': '1250000',
     'total': '1250000'},
    {'description': 'AI Development',
     'qty': '7',
     'price':